# Day 25 — 模拟交易第一周

**核心任务**：用 Day20-24 学的完整策略，在真实历史数据上跑模拟交易。

**策略速查**：MACD金叉买入 → 10天持仓 → -5%止损 → 仓位牛40%/震30%/熊20% → 盘前填计划卡

## 一、市场环境判断

先判断当前处于什么市场，决定仓位比例。

In [1]:
import pandas as pd
import numpy as np

STOCKS = {'招商银行': '/tmp/招商银行.csv', '茅台': '/tmp/茅台.csv'}

for name, path in STOCKS.items():
    df = pd.read_csv(path)
    df['日期'] = pd.to_datetime(df['日期'])
    df = df.set_index('日期').sort_index()
    d = df.iloc[-120:]
    
    d['MA20'] = d['收盘'].rolling(20).mean()
    d['MA60'] = d['收盘'].rolling(60).mean()
    ma_pct = (d['MA20'] > d['MA60']).sum() / len(d) * 100
    total_ret = (d['收盘'].iloc[-1]/d['收盘'].iloc[0]-1)*100
    
    if ma_pct > 50: regime, pos = '🟢牛市', '40%'
    elif ma_pct > 30: regime, pos = '🟡震荡', '30%'
    else: regime, pos = '🔴熊市', '20%'
    
    print(f'{name}: {d.index[0].date()}~{d.index[-1].date()} | 收益{total_ret:+.1f}%')
    print(f'  MA20>MA60占比: {ma_pct:.0f}% → {regime} → 仓位{pos}')
    print()

招商银行: 2026-01-09~2026-07-10 | 收益-6.1%
  MA20>MA60占比: 12% → 🔴熊市 → 仓位20%

茅台: 2026-01-09~2026-07-10 | 收益-13.4%
  MA20>MA60占比: 10% → 🔴熊市 → 仓位20%



## 二、模拟交易：招商银行

找到MACD金叉信号，填好交易计划卡，追踪每一天。

In [2]:
# 招商银行模拟交易
df = pd.read_csv('/tmp/招商银行.csv')
df['日期'] = pd.to_datetime(df['日期'])
df = df.set_index('日期').sort_index()

# 计算MACD
ema12 = df['收盘'].ewm(span=12, adjust=False).mean()
ema26 = df['收盘'].ewm(span=26, adjust=False).mean()
dif = ema12 - ema26
dea = dif.ewm(span=9, adjust=False).mean()

# 找有完整10天数据的金叉
signals = []
for i in range(1, len(df)):
    if dif.iloc[i] > dea.iloc[i] and dif.iloc[i-1] <= dea.iloc[i-1]:
        if i + 10 < len(df):
            signals.append(i)

# 取倒数第二个完整信号
si = signals[-2]
buy_price = df['收盘'].iloc[si]
capital = 100000
shares = int(capital * 0.20 / buy_price // 100 * 100)
stop_price = buy_price * 0.95

print(f'📋 交易计划卡')
print(f'买入日: {df.index[si].date()} | 价格: {buy_price:.2f}元')
print(f'数量: {shares}股 | 金额: {shares*buy_price:.0f}元 | 仓位: 20%')
print(f'止损: {stop_price:.2f}元 | 到期: {df.index[si+10].date()}')
print(f'最坏亏损: {(buy_price-stop_price)*shares:.0f}元')
print()

# 逐日追踪
print(f'{"日期":12s} {"收盘":>7s} {"盈亏%":>7s} {"盈亏":>8s} {"操作":15s}')
print('-' * 55)
for day in range(11):
    idx = si + day
    date = df.index[idx]
    price = df['收盘'].iloc[idx]
    ret = (price/buy_price-1)*100
    profit = (price-buy_price)*shares
    action = '📝买入' if day==0 else ('📤到期卖出' if day==10 else ('🛑止损!' if ret<=-5 else '——'))
    print(f'{str(date.date()):12s} {price:7.2f} {ret:+6.1f}% {profit:+8.0f}元 {action:15s}')

final_ret = (df['收盘'].iloc[si+10]/buy_price-1)*100
final_profit = (df['收盘'].iloc[si+10]-buy_price)*shares
print(f'\n结果: {final_ret:+.1f}% | 盈亏: {final_profit:+.0f}元 | 占本金: {final_profit/capital*100:+.2f}%')
print(f'\n无论盈亏，只要按计划执行，就是一笔好交易。')

📋 交易计划卡
买入日: 2026-04-16 | 价格: 38.91元
数量: 500股 | 金额: 19455元 | 仓位: 20%
止损: 36.96元 | 到期: 2026-04-30
最坏亏损: 973元

日期                收盘     盈亏%       盈亏 操作             
-------------------------------------------------------
2026-04-16     38.91   +0.0%       +0元 📝买入            
2026-04-17     38.52   -1.0%     -195元 ——             
2026-04-20     38.87   -0.1%      -20元 ——             
2026-04-21     38.87   -0.1%      -20元 ——             
2026-04-22     38.74   -0.4%      -85元 ——             
2026-04-23     38.81   -0.3%      -50元 ——             
2026-04-24     38.38   -1.4%     -265元 ——             
2026-04-27     38.38   -1.4%     -265元 ——             
2026-04-28     38.60   -0.8%     -155元 ——             
2026-04-29     37.50   -3.6%     -705元 ——             
2026-04-30     37.27   -4.2%     -820元 📤到期卖出          

结果: -4.2% | 盈亏: -820元 | 占本金: -0.82%

无论盈亏，只要按计划执行，就是一笔好交易。


## 三、发现的问题

**茅台太贵了**：1281元/股 × 100股(1手) = 128,158元。
本金10万，熊市仓位20%=2万，买不起一手。

**启示**：资金量决定选股范围。10万本金适合单价30-100元的股票。

## 四、今日总结

### 四大收获

1. **模拟和纸上谈兵完全不同**——看到每天盈亏数字跳动，才能真正感受损失厌恶
2. **最难的不是买入，是持有**——连续6天亏损时，每一天都想提前割
3. **资金量决定选股范围**——10万本金选30-100元区间的股票最灵活
4. **亏了≠做错了**——交易成功的标准是「是否执行了计划」，不是「是否赚钱」

### 完整策略（Day20-25）

```
买入: MACD金叉
持仓: 10个交易日 / -5%止损
仓位: 牛40% / 震30% / 熊20%
执行: 盘前填计划卡，盘中不碰鼠标
规则: T+1不追涨、跌停不重仓、低频省手续费
```